# 02 — HMM Model Selection via BIC

> **Goal:** Determine the optimal number of hidden states for the market regime HMM  
> using the Bayesian Information Criterion (BIC), then inspect the winning model.

**Sections**
1. Data + feature loading
2. BIC / AIC sweep (n_states = 2 → 5)
3. Best model — transition matrix, emission means, covariances
4. Regime-coloured S&P 500 price chart
5. Regime duration statistics
6. State interpretation

In [ ]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import yaml

from src.data.fetch import fetch_ohlcv
from src.features.engineer import build_feature_matrix
from src.model.train import model_selection, save_model
from src.model.predict import decode_regimes, predict_probabilities, label_regimes
from src.model.evaluate import regime_statistics, transition_matrix_display

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

REGIME_COLORS = {"Bull": "#16A34A", "Bear": "#DC2626", "Sideways": "#D97706"}

with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

df = fetch_ohlcv(
    config["data"]["ticker"],
    start=config["data"]["start_date"],
    end=config["data"]["end_date"],
)
features = build_feature_matrix(df, config)

print(f"OHLCV    : {df.shape}  [{df.index[0].date()} → {df.index[-1].date()}]")
print(f"Features : {features.shape}")
features.head()

---
## 1. BIC / AIC Sweep

We train a `GaussianHMM` for each candidate state count and compare information criteria.  
**BIC penalises model complexity more aggressively than AIC**, making it the primary selector.  
The elbow / minimum in the BIC curve indicates the best trade-off between fit and parsimony.

In [ ]:
results = model_selection(features, config)

best_model   = results["best_model"]
best_k       = results["best_n_states"]
bic_scores   = results["bic_scores"]
aic_scores   = results["aic_scores"]

print(f"\nBIC scores : {bic_scores}")
print(f"AIC scores : {aic_scores}")
print(f"\n→ Best model: k = {best_k} states (lowest BIC = {bic_scores[best_k]:,.2f})")

In [ ]:
ks   = sorted(bic_scores)
bics = [bic_scores[k] for k in ks]
aics = [aic_scores[k] for k in ks]

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(ks, bics, "o-", color="#2563EB", lw=2, ms=7, label="BIC")
ax.plot(ks, aics, "s--", color="#9333EA", lw=1.8, ms=6, label="AIC")
ax.axvline(best_k, color="#DC2626", lw=1.5, ls=":",
           label=f"Best k = {best_k} (BIC)")

ax.set_title("BIC / AIC vs. Number of Hidden States")
ax.set_xlabel("Number of states (k)")
ax.set_ylabel("Information Criterion")
ax.set_xticks(ks)
ax.legend(framealpha=0.85)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig("../outputs/07_bic_sweep.png", bbox_inches="tight")
plt.show()

---
## 2. Best Model Inspection

### 2a. Emission Means and Covariances

In [ ]:
regime_names = label_regimes(best_model)
print(f"State → regime mapping: {regime_names}\n")

feat_cols = features.columns.tolist()

means_df = pd.DataFrame(
    best_model.means_,
    index=[f"State {i} ({regime_names[i]})" for i in range(best_model.n_components)],
    columns=feat_cols,
)
print("Emission means:")
print(means_df.round(5).to_string())
print()

for i in range(best_model.n_components):
    cov = pd.DataFrame(
        best_model.covars_[i],
        index=feat_cols, columns=feat_cols,
    )
    print(f"Covariance — State {i} ({regime_names[i]}):")
    print(cov.round(6).to_string())
    print()

### 2b. Transition Matrix

In [ ]:
trans = transition_matrix_display(best_model, regime_names)
print("Transition matrix (rows = from, columns = to):")
print(trans.to_string())
print()

# Implied mean regime duration = 1 / (1 - P_ii) trading days
print("Implied mean duration per regime:")
for name in trans.columns:
    p_stay = trans.loc[name, name]
    if p_stay < 1.0:
        mean_dur = 1.0 / (1.0 - p_stay)
        print(f"  {name:10s}: {mean_dur:.1f} trading days  (P_stay = {p_stay:.4f})")

---
## 3. Regime-Coloured S&P 500 Price Chart

In [ ]:
labels = decode_regimes(best_model, features)

# Align price index with feature index (features start 21 days later)
price = df["Close"].loc[features.index]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})

# ── top panel: price coloured by regime ──────────────────────────────────
ax = axes[0]
for state_id, name in regime_names.items():
    mask = labels == state_id
    ax.fill_between(
        price.index, price.min() * 0.95, price.max() * 1.05,
        where=mask, alpha=0.18,
        color=REGIME_COLORS.get(name, "#94A3B8"),
    )

ax.plot(price.index, price, color="#1E293B", lw=0.9, zorder=3)
ax.set_ylabel("S&P 500 Close (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title(f"S&P 500 — HMM Regime Classification (k={best_k})")

legend_patches = [
    mpatches.Patch(
        color=REGIME_COLORS.get(name, "#94A3B8"), alpha=0.5,
        label=name,
    )
    for name in regime_names.values()
]
ax.legend(handles=legend_patches, loc="upper left", framealpha=0.85)

# ── bottom panel: regime label bar ───────────────────────────────────────
ax2 = axes[1]
regime_int = labels.map({v: i for i, v in enumerate(["Bear", "Sideways", "Bull"])})
ax2.scatter(labels.index, regime_int, c=labels.map(
    {sid: REGIME_COLORS.get(name, "#94A3B8") for sid, name in regime_names.items()}
), s=1.5, linewidths=0)
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(["Bear", "Sideways", "Bull"], fontsize=9)
ax2.set_ylabel("Regime")
ax2.set_xlabel("")

plt.tight_layout()
plt.savefig("../outputs/08_regime_chart.png", bbox_inches="tight")
plt.show()

---
## 4. Regime Duration Statistics

In [ ]:
stats = regime_statistics(labels, regime_names)
print("Regime statistics:")
print(stats.to_string())
stats

---
## 5. State Interpretation

In [ ]:
print("State interpretation based on emission means:\n")
for i in range(best_model.n_components):
    name = regime_names[i]
    mean_ret = best_model.means_[i, 0]
    mean_vol = best_model.means_[i, 1]
    mean_vr  = best_model.means_[i, 2]
    ann_ret  = mean_ret * 252
    print(
        f"  State {i} → {name:10s} | "
        f"mean log_return = {mean_ret:+.5f}  ({ann_ret:+.1%} ann.) | "
        f"realized_vol = {mean_vol:.3f}  ({mean_vol*100:.1f}% ann.) | "
        f"volume_ratio = {mean_vr:.3f}"
    )

---
## 6. Save Best Model

In [ ]:
save_model(best_model, f"../models/hmm_{best_k}state.pkl")
print(f"Saved best model (k={best_k}) → models/hmm_{best_k}state.pkl")

---
## Summary

| Finding | Detail |
|---------|--------|
| **Optimal k** | Determined by BIC minimum — see elbow plot above |
| **Bull regime** | Highest mean log_return, lowest volatility, normal volume |
| **Bear regime** | Lowest (most negative) mean log_return, highest volatility, elevated volume |
| **Sideways regime** | Near-zero mean return, intermediate vol — drift/consolidation phase |
| **Persistence** | Diagonal transition probabilities >> 0.9 confirm regimes are sticky |
| **BIC formula** | `-2·LL + p·ln(n)` where `p = k²-1 + k·d + k·d·(d+1)/2` |

*Model saved to `models/`.  Proceed to API + dashboard phase.*